# XRD Rietveld Plot Generator

Publication-quality Rietveld plots from the **CSV that the GSAS-II Rietveld
plot saves** - batch processing, built-in validation, cross-platform.

Not the file from *Export → Powder data as → histogram CSV file*: that one
has a quoted preamble and different column names, and is rejected.

Full documentation (input format, usage, configuration, privacy notes):
see [`README.md`](README.md).

## 1. Setup

Dependency check, then the engine. Parsing, data preparation, plotting and
the batch driver live in [`xrd_plotter.py`](xrd_plotter.py), imported here
as `xp`. Plot appearance (2θ window, colours, line widths, fonts) is set by
the constants at the top of that file and overridden on the module, as the
cell below shows. Input format and numerical-precision details are in the
README.

In [ ]:
# Dependency bootstrap - installs only what is missing. IPython arrives with
# any Jupyter kernel, and is listed so an editor resolves it as well.
import importlib.util, subprocess, sys

for module, package in (("numpy", "numpy"), ("pandas", "pandas"),
                        ("matplotlib", "matplotlib"),
                        ("ipywidgets", "ipywidgets"), ("IPython", "ipython"),
                        ("pytest", "pytest")):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "--quiet", package])
print("Dependencies OK")

In [ ]:
"""The plotting engine lives in xrd_plotter.py; this cell loads it."""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Figures appear under the cell drawing them. Set explicitly: an editor
# other than VS Code might start on a different backend.
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # type: ignore[name-defined]
except NameError:
    pass  # plain python, no IPython running

import xrd_plotter as xp

# Appearance is set by the constants in the module. Override them here, on
# the module itself, so every routine sees the change:
#   xp.PLOT_X_MIN, xp.PLOT_X_MAX = 13, 85   # fix the 2theta window
#   xp.WEIGHTED_RESIDUALS = False           # raw diff in the lower panel
#   xp.PHASE_COLORS = {"phase 1": "#1f77b4"}
print("Engine loaded:", Path(xp.__file__).name)


## 2. Validation (self-test on synthetic data)

Runs [`test_xrd_plotter.py`](test_xrd_plotter.py). The suite rebuilds
synthetic GSAS-II-style exports and asserts bit-exact parsing in both
separator and decimal-mark variants, detection of the phase columns among a
full set of export columns, isolation of corrupt, ragged and incomplete
files, the order and colours of the phases, the metadata binding into the
legend, the 2theta window, the unweighted residual panel, the interactive
helper, and a batch which survives one unusable file and repeats it in the
summary.

The cell fails on the first failing assertion, so running the notebook is a
test run, and so is `pytest -q` from a terminal. Only synthetic data is
used.

In [ ]:
import pytest

# The suite builds its own synthetic exports, so this cell reads nothing
# from data/ and works on a fresh clone.
exit_code = pytest.main(["-q", "--no-header", "test_xrd_plotter.py"])
assert exit_code == 0, "the validation suite failed, see the report above"
print("\nALL VALIDATION CHECKS PASSED")

## 3. Plot your own exports

Copy your CSV exports into `data/`, optionally place `Samples_metadata.csv`
next to the notebook, set the four values below and run.

Each file gets one block: its name, the phases found with the colour each
one was drawn in, the figure, then the two files written to `output/`. A
file the engine fails to draw prints `FAILED` with the reason and the run
continues. Every failure is repeated in the summary at the end.

> **Keep your data private:** `data/`, `output/` and `Samples_metadata.csv`
> are listed in `.gitignore` and must never be committed or uploaded.

In [ ]:
DATA_FOLDER = "data"                       # your GSAS-II CSV exports
METADATA_FILE = "Samples_metadata.csv"     # optional, PRIVATE - never commit
OUTPUT_FOLDER = "output"                   # created automatically
USE_SQRT = True                            # False -> linear intensity axis

results = xp.process_folder(DATA_FOLDER, METADATA_FILE, OUTPUT_FOLDER,
                         use_sqrt=USE_SQRT)

## 4. Try a different window on one file

Pick a file and type the limits. The figure below redraws in place on every
change: a box on Enter or when you leave it, a checkbox and the dropdown at
once. **Redraw** repeats it on demand. Nothing is saved here, so this is
where you settle on a window before running section 3 again.

An empty box leaves its end of the axis to the setting behind it, the
`xp.PLOT_X_MIN` and `xp.PLOT_X_MAX` constants for 2θ and the data itself for
the intensity.

The line above the figure holds the engine messages for this file and the
metadata row for the window on screen. Paste the row into
`Samples_metadata.csv` and section 3 draws this sample this way every time.

This section needs `ipywidgets`, which the first cell installs. Without it
the section prints how to install it and the rest of the notebook is
unaffected.

In [ ]:
# No widgets.Output here. An Output cleared inside a callback appends a second
# figure under the first in VS Code, so the panel would fill with stale plots.
# The figure is a widgets.Image and the log a widgets.HTML, both value
# replaced, so every redraw updates the same two areas in place.
import contextlib
import html
import io
import traceback

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    print("ipywidgets is not installed: run 'pip install ipywidgets', "
          "then re-run this cell.")

files = sorted(f for f in Path(DATA_FOLDER).glob("*.csv")
               if f.name != Path(METADATA_FILE).name)


def pre(text):
    """Escaped monospace block for a widgets.HTML value."""
    return ("<pre style='margin:0;font:12px/1.4 monospace;white-space:pre-wrap'>"
            f"{html.escape(text)}</pre>")


if widgets is None or not files:
    if widgets is not None:
        print(f"No CSV files in '{DATA_FOLDER}': nothing to replot.")
else:
    picker = widgets.Dropdown(options=[(f.name, str(f)) for f in files],
                              description="File:",
                              layout=widgets.Layout(width="420px"))
    # continuous_update=False: a box redraws when you press Enter or leave
    # it, not on every keystroke.
    boxes = {k: widgets.Text(description=d, placeholder="auto",
                             continuous_update=False,
                             layout=widgets.Layout(width="180px"))
             for k, d in (("x_min", "2theta min"), ("x_max", "2theta max"),
                          ("y_min", "y min"), ("y_max", "y max"))}
    sqrt_box = widgets.Checkbox(value=USE_SQRT, description="sqrt intensity")
    weighted_box = widgets.Checkbox(value=xp.WEIGHTED_RESIDUALS,
                                    description="diff/sigma")
    redraw_button = widgets.Button(description="Redraw", button_style="primary")
    status = widgets.HTML()
    canvas = widgets.Image(format="png",
                           layout=widgets.Layout(width="100%",
                                                 max_width="820px"))
    busy = [False]

    def redraw(_=None):
        """Draw the picked file with whatever the controls now hold."""
        if busy[0]:
            return  # ignore events queued while a redraw is running
        busy[0] = True
        try:
            limits = {k: xp.to_number(b.value) if b.value.strip() else None
                      for k, b in boxes.items()}
            log = io.StringIO()
            try:
                # Engine messages belong in the status block. Left on stdout
                # they land under the cell and pile up one copy per redraw.
                with contextlib.redirect_stdout(log):
                    fig, line = xp.replot_file(picker.value, METADATA_FILE,
                                               use_sqrt=sqrt_box.value,
                                               weighted=weighted_box.value,
                                               **limits)
            except ValueError as e:
                # Hide the stale figure: it belongs to a different file, and
                # leaving it up invites new limits typed against it.
                canvas.layout.display = "none"
                status.value = pre(f"{log.getvalue()}Cannot draw this file: {e}")
                return
            png = io.BytesIO()
            try:
                # 110 dpi is a screen preview. The batch writes the 600 dpi file.
                fig.savefig(png, format="png", dpi=110, bbox_inches="tight",
                            facecolor="white")
            finally:
                plt.close(fig)  # a failed render must not leak the figure
            canvas.value = png.getvalue()
            canvas.layout.display = ""
            status.value = pre(f"{log.getvalue()}"
                               f"Metadata line for this window:\n{line}")
        except Exception:
            # Same guard xrd_plotter.py uses in its batch loop: create_plot's
            # figure must not leak if it raised after opening it.
            plt.close("all")
            canvas.layout.display = "none"
            status.value = pre("Redraw failed:\n" + traceback.format_exc())
        finally:
            busy[0] = False

    for control in (picker, sqrt_box, weighted_box, *boxes.values()):
        control.observe(redraw, names="value")
    redraw_button.on_click(redraw)

    display(widgets.VBox([
        picker,
        widgets.HBox([boxes["x_min"], boxes["x_max"]]),
        widgets.HBox([boxes["y_min"], boxes["y_max"]]),
        widgets.HBox([sqrt_box, weighted_box, redraw_button]),
        status,
        canvas,
    ]))
    redraw()  # open on the first file instead of an empty panel
